# Lab 4, Bronze ingestion

I'm using Databricks' built in retail-org sample dataset for this lab instead of the e-commerce data from the team project, since this one is basically made for practicing SCD and schema evolution (the customers file already has valid_from, valid_to and loyalty_segment columns in it).

First I just want to poke around the raw files a bit before writing anything, so I know what I'm actually working with.

In [0]:
%fs ls /databricks-datasets/retail-org/

In [0]:
%fs ls /databricks-datasets/retail-org/customers/

In [0]:
%fs ls /databricks-datasets/retail-org/sales_orders/

A quick look at both files before deciding how to read them properly:

In [0]:
display(spark.read.option("header", "true").option("inferSchema", "true")
    .csv("dbfs:/databricks-datasets/retail-org/customers/*.csv"))

In [0]:
display(spark.read.json("dbfs:/databricks-datasets/retail-org/sales_orders/*.json"))

In [0]:
%fs head dbfs:/databricks-datasets/retail-org/sales_orders/part-00000

## Setting up the catalog and schemas

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS lab4;
CREATE SCHEMA IF NOT EXISTS lab4.bronze;
CREATE SCHEMA IF NOT EXISTS lab4.silver;
CREATE VOLUME IF NOT EXISTS lab4.bronze.checkpoints;

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "lab4", "Catalog")
catalog = dbutils.widgets.get("catalog")

## Loading customers into bronze

Loading it exactly as is, duplicates and all. I'm not cleaning anything here on purpose, dedup and history tracking belong in silver, not bronze. Using overwrite mode so this notebook can be rerun any time and always ends up in the same state.


In [0]:
cust_raw = (
    spark.read
    .option("header", "true")
    .csv("dbfs:/databricks-datasets/retail-org/customers/*.csv")
    .withColumn("source", F.lit("retail-org/customers"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

(cust_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.bronze.brz_customers")
)

print(f"Loaded {cust_raw.count()} customer rows into bronze")

Checking straight away if there are duplicate customers in the raw file, just so I know for sure I actually need a dedup step in silver later, instead of assuming it.

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS distinct_customers,
    COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_rows
FROM lab4.bronze.brz_customers

## Loading sales orders into bronze

This one's nested JSON, each order has an array of products inside it, and each product even has its own nested promo info. I'm keeping it nested here on purpose, flattening it out is a silver problem, not a bronze one.


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

promotion_info_schema = StructType([
    StructField("promo_id", StringType()),
    StructField("promo_disc", StringType()),
])

ordered_product_schema = StructType([
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("price", StringType()),
    StructField("qty", StringType()),
    StructField("unit", StringType()),
    StructField("curr", StringType()),
    StructField("promotion_info", promotion_info_schema),
])

orders_raw_schema = StructType([
    StructField("order_number", StringType()),
    StructField("customer_id", StringType()),
    StructField("customer_name", StringType()),
    StructField("order_datetime", StringType()),
    StructField("number_of_line_items", StringType()),
    StructField("ordered_products", ArrayType(ordered_product_schema)),
])

orders_raw = (
    spark.read
    .schema(orders_raw_schema)
    .json("dbfs:/databricks-datasets/retail-org/sales_orders/*.json")
    .withColumn("source", F.lit("retail-org/sales_orders"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

(orders_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.bronze.brz_sales_orders")
)

print(f"Loaded {orders_raw.count()} sales order rows into bronze")

Quick look at both schemas before moving on to silver.

In [0]:
%sql
DESCRIBE TABLE lab4.bronze.brz_customers

In [0]:
%sql
DESCRIBE TABLE lab4.bronze.brz_sales_orders